# AI Code Provenance & Security Intelligence Lab

This notebook estimates statistical consistency with human, AI and hybrid
authorship while preserving an explicit unknown/OOD outcome. It separates
authorship, public reuse and security risk; none is treated as proof of another.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from code_provenance.repository import working_tree_samples, recent_commit_metadata
from code_provenance.features import extract_features
from code_provenance.reuse import PublicReuseIndex
from code_provenance.security import scan_code, python_dependencies
from code_provenance.report import descriptive_repository_report


## Part 1 — Claim contract and target variables

The target is a calibrated distribution over human, AI and hybrid classes, plus
abstention. “Organic fraction” is a declared index and is hidden whenever the
sample is uncertain or out of distribution.


In [ ]:
claim_contract = pd.DataFrame([
 {'quantity':'authorship probability','meaning':'similarity to verified training distributions','not_allowed':'proof of author identity'},
 {'quantity':'public reuse fraction','meaning':'token-shingle overlap with indexed corpus','not_allowed':'AI attribution'},
 {'quantity':'organic fraction','meaning':'human/hybrid probability discounted by reuse','not_allowed':'literal keystroke percentage'},
 {'quantity':'security signal','meaning':'review surface','not_allowed':'confirmed vulnerability without validation'},
])
display(claim_contract)


## Part 2 — Read-only repository evidence

Tracked files and Git metadata are extracted without running target code. No
authorship label is inferred from author names, emails or commit style.


In [ ]:
report = descriptive_repository_report(ROOT)
display(pd.Series(report, name='repository evidence'))
samples = working_tree_samples(ROOT)
commits = pd.DataFrame(recent_commit_metadata(ROOT, limit=100))
display(commits.head())


## Part 3 — Code DNA: lexical and AST evidence

Features are interpretable and exclude repository identity. Python receives AST
features; other languages use conservative lexical structure until Tree-sitter
parsers are versioned and validated.


In [ ]:
feature_frame = pd.DataFrame([{'path': s.path, 'language': s.language, **extract_features(s)} for s in samples])
display(feature_frame.head())
display(feature_frame.describe().T)


## Part 4 — Public-code reuse as a separate channel

Build the index only from a corpus with recorded URL, revision and license.
Overlap does not identify whether a human or model performed the reuse.


In [ ]:
reuse = PublicReuseIndex(width=7)
# reuse.add(public_corpus_documents)  # enable only after provenance/licence audit
print({'indexed_documents': reuse.documents, 'status': 'EMPTY_UNTIL_AUDITED_PUBLIC_CORPUS'})


## Part 5 — Security and supply-chain evidence

Local patterns are screening signals. Production comparisons require CodeQL or
Semgrep/Bandit and expert confirmation. Registry outage must remain `unknown`.


In [ ]:
security_rows = []
for sample in samples:
    for signal in scan_code(sample.code):
        security_rows.append({'path':sample.path, **signal.__dict__})
security = pd.DataFrame(security_rows)
display(security.head(20) if not security.empty else pd.DataFrame({'status':['no local pattern signals']}))


## Part 6 — Verified corpus and group-disjoint training

Training is intentionally unavailable until `data/code_provenance/manifest.csv`
contains admissible labels. Repository/author groups and near-duplicate clusters
must remain disjoint. Random snippet splits are prohibited.


In [ ]:
manifest = ROOT / 'data' / 'code_provenance' / 'manifest.csv'
training_status = 'READY' if manifest.exists() else 'BLOCKED_NO_VERIFIED_LABEL_MANIFEST'
print({'manifest': str(manifest), 'training_status': training_status})


## Part 7 — Calibration, hybrid authorship and OOD abstention

The classifier reports group-out-of-fold macro-F1 and calibrated probabilities.
Unseen language, repository domain or generator evaluations are separate tests.
Low confidence or high OOD score produces `unknown`, not a forced binary label.


## Part 8 — Security-controlled comparison

Compare human, AI, AI→human and human→AI changes only after matching language,
repository, change size, complexity, file role and project age. Use security
findings per 1,000 changed lines and repository-clustered confidence intervals.


## Part 9 — Adaptive review policy

The later decision layer chooses among allow, cheap scan, deep static analysis,
LLM security review, human review and block. It optimizes findings retained
subject to compute, reviewer-time and developer-friction constraints. Policy
learning begins only after real review outcomes exist.


## Part 10 — Current conclusion and next evidence gate

The extraction, feature, reuse, security, dataset and modelling contracts are
implemented. This repository has only descriptive evidence today; it has no
defensible AI/organic percentage until a verified group-disjoint corpus is
acquired and evaluated. That refusal is the intended scientific behavior.
